In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")


## Data

`n_samples` is per optimizer. The plots below therefore work from the summary table for the full search, and show fold-level detail only for the top configurations.

In [ ]:
folds = pd.read_csv("../out/gridsearch/folds.csv")
results = pd.read_csv("../out/gridsearch/results.csv")

optimizer_order = sorted(results["optimizer"].unique())
palette = dict(zip(optimizer_order, sns.color_palette("colorblind", len(optimizer_order))))

summary_by_optimizer = (
    results.groupby("optimizer")
    .agg(
        n_configs=("config_id", "size"),
        mean_accuracy=("mean_val_accuracy", "mean"),
        best_accuracy=("mean_val_accuracy", "max"),
        median_fold_time_s=("mean_total_time_s", "median"),
    )
    .reindex(optimizer_order)
)

print(f"{len(results):,} configurations, {len(folds):,} fold rows")
summary_by_optimizer.round(4)


## Search overview

The first panel is deliberately limited to the strongest configurations; the remaining panels retain every configuration.

In [ ]:
top_n = min(25, len(results))
top = results.nlargest(top_n, "mean_val_accuracy").sort_values("mean_val_accuracy")
top = top.assign(label=top.apply(lambda row: f"{row.config_id} ({row.optimizer})", axis=1))

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)

for optimizer, group in top.groupby("optimizer", sort=False):
    axes[0, 0].errorbar(
        group["mean_val_accuracy"], group["label"], xerr=group["std_val_accuracy"],
        fmt="o", capsize=3, color=palette[optimizer], label=optimizer,
    )
axes[0, 0].set(title=f"Top {top_n} configurations (mean ± fold SD)", xlabel="validation accuracy", ylabel="config_id (optimizer)")
axes[0, 0].legend(title="optimizer")

sns.boxplot(data=results, x="optimizer", y="mean_val_accuracy", order=optimizer_order, color="lightsteelblue", ax=axes[0, 1])
sns.stripplot(data=results, x="optimizer", y="mean_val_accuracy", order=optimizer_order, color="black", alpha=0.22, size=2, ax=axes[0, 1])
axes[0, 1].set(title="All configuration scores by optimizer", xlabel="optimizer", ylabel="mean validation accuracy")

for optimizer, group in results.groupby("optimizer", sort=False):
    axes[1, 0].scatter(group["mean_total_time_s"], group["mean_val_accuracy"], s=20, alpha=0.55, color=palette[optimizer], label=optimizer)
axes[1, 0].set(title="Accuracy versus time (all configurations)", xlabel="mean time per fold (s)", ylabel="mean validation accuracy")
axes[1, 0].legend(title="optimizer")

top_folds = folds[folds["config_id"].isin(top["config_id"])].copy()
labels_by_config = top.set_index("config_id")["label"]
top_folds["label"] = top_folds["config_id"].map(labels_by_config)
fold_order = top["label"].tolist()
sns.boxplot(data=top_folds, y="label", x="val_accuracy", order=fold_order, color="lightsteelblue", ax=axes[1, 1])
sns.stripplot(data=top_folds, y="label", x="val_accuracy", order=fold_order, color="black", size=3, ax=axes[1, 1])
axes[1, 1].set(title=f"Fold variation for the top {top_n}", xlabel="validation accuracy", ylabel="config_id")

plt.show()


## Hyperparameters versus accuracy

Each point is one cross-validated configuration. Colour makes it possible to see whether a parameter range behaves differently for an optimizer.

In [ ]:
params = [
    "eta_infer", "T_infer", "lr", "weight_decay",
    "negative_slope", "batch_size", "hidden_width", "n_hidden_layers",
]
log_params = {"eta_infer", "lr", "weight_decay", "negative_slope", "batch_size", "hidden_width"}

fig, axes = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)
for ax, param in zip(axes.ravel(), params):
    for optimizer, group in results.groupby("optimizer", sort=False):
        ax.scatter(group[param], group["mean_val_accuracy"], s=18, alpha=0.5, color=palette[optimizer], label=optimizer)
    if param in log_params:
        ax.set_xscale("log")
    ax.set(xlabel=param, ylabel="mean validation accuracy")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="optimizer", loc="upper center", ncol=len(optimizer_order), bbox_to_anchor=(0.5, 1.06))
plt.show()


## Best configurations


In [ ]:
columns = [
    "config_id", "optimizer", "dims", "eta_infer", "T_infer", "lr", "weight_decay",
    "negative_slope", "batch_size", "mean_val_accuracy", "std_val_accuracy",
    "mean_val_f1_macro", "mean_total_time_s",
]
results.nlargest(20, "mean_val_accuracy")[columns].round(4)
